In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


What is Gaussian Capula? How it is being used in our pipeline?


 A Gaussian Copula is a statistical tool that separates two properties
 of multivariate data:
   1. Each variable's individual distribution (marginals)
   2. The correlation structure between variables

 It takes any dataset and impose a desired correlation structure
 onto it, while preserving each variable's own distribution.

 WHY DO WE NEED IT?
 Our cGAN generates synthetic Genomaps that are visually realistic
 and class-specific. However, when reversed back to gene expression
 space, the gene-gene correlation structure is lost (Pearson r ≈ 0).
 The cGAN learns spatial image patterns, not gene co-expression.
 The Gaussian Copula fixes this as a post-processing step.

 HOW WE IMPLEMENT IT (4 steps):
 Step 1: Rank-transform each gene in synthetic data → uniform [0,1]
 Step 2: Map uniform → Gaussian via inverse CDF (norm.ppf)
 Step 3: Impose real data correlation via Cholesky decomposition
 Step 4: Map back to real data marginals via quantile matching

 WHY IS THIS SCIENTIFICALLY VALID?
 The reverse transformation already uses real data statistics:
   - T matrix (from real gene-gene interactions)
   - original_means and original_stds (from real data)
 The copula simply adds the correlation matrix to this — consistent
 with the same philosophy of using real data as a biological reference

In [ ]:
# ============================================================
# Generate Synthetic Gene Expression — Corrected Pipeline
# PBMC Dataset
# Save to new location
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import tensorflow as tf
import os

BASE      = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
REV_DOCS  = BASE + '/Reverse GeneExpression Required Docs'
SAVE_DIR  = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Load everything ----
print("Loading real genomaps to compute _MIN/_MAX...")
b_real    = np.load(BASE + '/b_Class_genomap.npy').astype(np.float32)
mono_real = np.load(BASE + '/mono_Class_genomap.npy').astype(np.float32)
if b_real.ndim == 3:    b_real    = b_real[..., np.newaxis]
if mono_real.ndim == 3: mono_real = mono_real[..., np.newaxis]

stacked = np.vstack([b_real, mono_real])
_MIN = float(stacked.min())
_MAX = float(stacked.max())
print(f"_MIN={_MIN:.6f} | _MAX={_MAX:.6f}")

T_b    = np.load(REV_DOCS + '/transformation_matrix_T_b_class.npy')
T_mono = np.load(REV_DOCS + '/transformation_matrix_T_mono_class.npy')
means_b    = np.load(REV_DOCS + '/original_means_b_class.npy')
means_mono = np.load(REV_DOCS + '/original_means_mono_class.npy')
stds_b     = np.load(REV_DOCS + '/original_stds_b_class.npy')
stds_mono  = np.load(REV_DOCS + '/original_stds_mono_class.npy')

num_genes      = T_b.shape[1]
totalGridPoint = 40 * 40
print(f"num_genes={num_genes} | totalGridPoint={totalGridPoint}")

# ---- Load generator ----
print("\nLoading generator...")
generator = tf.keras.models.load_model(BASE + '/generator.h5')

# ---- Generate synthetic genomaps ----
n_b    = b_real.shape[0]     # 1186
n_mono = mono_real.shape[0]  # 1186
latent_dim = 128

print(f"Generating {n_b} synthetic B genomaps    (label=-0.5)...")
noise_b  = tf.random.normal(shape=(n_b, latent_dim))
labels_b = np.full((n_b, 1), -0.5, dtype=np.float32)
syn_b_tanh = generator.predict([noise_b, labels_b], batch_size=64)

print(f"Generating {n_mono} synthetic Mono genomaps (label=+0.5)...")
noise_mono  = tf.random.normal(shape=(n_mono, latent_dim))
labels_mono = np.full((n_mono, 1), 0.5, dtype=np.float32)
syn_mono_tanh = generator.predict([noise_mono, labels_mono], batch_size=64)

print(f"syn_b_tanh:   {syn_b_tanh.shape}  range=[{syn_b_tanh.min():.3f}, {syn_b_tanh.max():.3f}]")
print(f"syn_mono_tanh: {syn_mono_tanh.shape} range=[{syn_mono_tanh.min():.3f}, {syn_mono_tanh.max():.3f}]")

# ---- Reverse min-max normalization ----
def reverse_minmax(syn_tanh, _MIN, _MAX):
    return ((syn_tanh + 1) * (_MAX - _MIN) / 2) + _MIN

syn_b_orig    = reverse_minmax(syn_b_tanh,    _MIN, _MAX)
syn_mono_orig = reverse_minmax(syn_mono_tanh, _MIN, _MAX)
print(f"\nAfter min-max reversal:")
print(f"  syn_b range:   [{syn_b_orig.min():.4f}, {syn_b_orig.max():.4f}]")
print(f"  syn_mono range: [{syn_mono_orig.min():.4f}, {syn_mono_orig.max():.4f}]")

# ---- Option B: corrected pseudo-inverse reverse ----
def reverse_optionB(syn_orig_scale, T, means, stds):
    n_samples   = syn_orig_scale.shape[0]
    projMat     = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)
    projM = np.zeros((n_samples, num_genes), dtype=np.float32)
    for i in range(n_samples):
        ex = syn_orig_scale[i, :, :, 0].flatten(order='F')
        projM[i, :] = ex[:num_genes]
    X_norm = np.matmul(projM, projMat_inv)
    return ((X_norm * stds) + means).astype(np.float32)

print("\nApplying corrected pseudo-inverse reverse...")
syn_b_expr    = reverse_optionB(syn_b_orig,   T_b,    means_b,    stds_b)
syn_mono_expr = reverse_optionB(syn_mono_orig, T_mono, means_mono, stds_mono)

print(f"syn_b_expr:    {syn_b_expr.shape}  range=[{syn_b_expr.min():.3f}, {syn_b_expr.max():.3f}]")
print(f"syn_mono_expr: {syn_mono_expr.shape} range=[{syn_mono_expr.min():.3f}, {syn_mono_expr.max():.3f}]")

# ---- Save ----
b_save_path    = SAVE_DIR + '/new_recovered_b_expression.csv'
mono_save_path = SAVE_DIR + '/new_recovered_mono_expression.csv'

pd.DataFrame(syn_b_expr).to_csv(   b_save_path,    index=False, header=False, float_format='%.6f')
pd.DataFrame(syn_mono_expr).to_csv(mono_save_path, index=False, header=False, float_format='%.6f')

print(f"\nSaved:")
print(f"  {b_save_path}")
print(f"  {mono_save_path}")
print("Done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading real genomaps to compute _MIN/_MAX...
_MIN=-2.464965 | _MAX=34.409313
num_genes=1600 | totalGridPoint=1600

Loading generator...


Generating 1186 synthetic B genomaps    (label=-0.5)...
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step
Generating 1186 synthetic Mono genomaps (label=+0.5)...
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 130ms/step
syn_b_tanh:   (1186, 40, 40, 1)  range=[-0.971, 0.836]
syn_mono_tanh: (1186, 40, 40, 1) range=[-0.966, 0.672]

After min-max reversal:
  syn_b range:   [-1.9350, 31.3866]
  syn_mono range: [-1.8365, 28.3528]

Applying corrected pseudo-inverse reverse...
syn_b_expr:    (1186, 1600)  range=[-2.209, 66.199]
syn_mono_expr: (1186, 1600) range=[-2.914, 123.980]

Saved:
  /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_b_expression.csv
  /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_mono_expression.csv
Done.


In [ ]:
# ============================================================
# Gaussian Copula Post-processing + 3 Metrics + PDF/CDF
# PBMC Dataset
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, spearmanr, pearsonr, mannwhitneyu
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

BASE     = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
SAVE_DIR = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Reverse_PBMC_Biological_Fidelity_Analysis'

# ============================================================
# 1. LOAD DATA
# ============================================================
print("Loading data...")
real_b_expr    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',    header=None).values.astype(np.float32)
real_mono_expr = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv', header=None).values.astype(np.float32)

# Load Option B synthetic gene expression (best from previous run)
syn_b_optB    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_b_expression.csv', header=None).values.astype(np.float32)
syn_mono_optB = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_mono_expression.csv', header=None).values.astype(np.float32)

print(f"real_b:    {real_b_expr.shape}  | real_mono:    {real_mono_expr.shape}")
print(f"syn_b_B:   {syn_b_optB.shape}   | syn_mono_B:   {syn_mono_optB.shape}")

Loading data...
real_b:    (1186, 1600)  | real_mono:    (1186, 1600)
syn_b_B:   (1186, 1600)   | syn_mono_B:   (1186, 1600)


In [ ]:
# ============================================================
# Gaussian Copula Post-processing + 3 Metrics + PDF/CDF
# PBMC Dataset
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, spearmanr, pearsonr, mannwhitneyu
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

BASE     = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
SAVE_DIR = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Reverse_PBMC_Biological_Fidelity_Analysis'

# ============================================================
# 1. LOAD DATA
# ============================================================
print("Loading data...")
real_b_expr    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',    header=None).values.astype(np.float32)
real_mono_expr = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv', header=None).values.astype(np.float32)

# Load Option B synthetic gene expression (best from previous run)
syn_b_optB    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_b_expression.csv', header=None).values.astype(np.float32)
syn_mono_optB = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/new_recovered_mono_expression.csv', header=None).values.astype(np.float32)

print(f"real_b:    {real_b_expr.shape}  | real_mono:    {real_mono_expr.shape}")
print(f"syn_b_B:   {syn_b_optB.shape}   | syn_mono_B:   {syn_mono_optB.shape}")

# ============================================================
# 2. GAUSSIAN COPULA POST-PROCESSING
# ============================================================
print("\nApplying Gaussian Copula...")

def apply_gaussian_copula(syn_expr, real_expr):
    """
    Transform synthetic gene expression to match real correlation structure
    while preserving marginal distributions of real data.

    Steps:
    1. Rank-transform synthetic → uniform [0,1] per gene
    2. Map uniform → Gaussian via norm.ppf
    3. Impose real correlation via Cholesky decomposition
    4. Map back to original marginals using real data quantiles
    """
    n_syn, n_genes = syn_expr.shape

    # Step 1: Rank-transform synthetic to uniform pseudo-observations
    ranks = np.argsort(np.argsort(syn_expr, axis=0), axis=0)
    u = (ranks + 1) / (n_syn + 1)          # shape: (n_syn, n_genes), values in (0,1)

    # Step 2: Map uniform → standard Gaussian
    z = norm.ppf(u)                          # shape: (n_syn, n_genes)

    # Step 3: Compute real correlation matrix and Cholesky decomposition
    R_real = np.corrcoef(real_expr.T)        # (n_genes, n_genes)
    # Add small jitter for numerical stability
    R_real += np.eye(n_genes) * 1e-6
    L = np.linalg.cholesky(R_real)           # lower triangular Cholesky factor

    # Standardize z (zero mean, unit variance per gene)
    z = (z - z.mean(axis=0)) / (z.std(axis=0) + 1e-8)

    # Apply Cholesky to impose real correlation structure
    z_corr = z @ L.T                         # (n_syn, n_genes)

    # Step 4: Map back to real marginals via quantile matching
    u_corr = norm.cdf(z_corr)               # back to uniform (0,1)
    u_corr = np.clip(u_corr, 0.001, 0.999)

    # For each gene, map quantiles to real data values
    syn_copula = np.zeros_like(syn_expr)
    for g in range(n_genes):
        syn_copula[:, g] = np.quantile(real_expr[:, g], u_corr[:, g])

    return syn_copula.astype(np.float32)

syn_b_copula    = apply_gaussian_copula(syn_b_optB,    real_b_expr)
syn_mono_copula = apply_gaussian_copula(syn_mono_optB, real_mono_expr)

print(f"Copula output — syn_b:    {syn_b_copula.shape}")
print(f"Copula output — syn_mono: {syn_mono_copula.shape}")

# Save copula outputs
pd.DataFrame(syn_b_copula).to_csv(   SAVE_DIR + '/copula_recovered_b_expression.csv',    index=False, header=False)
pd.DataFrame(syn_mono_copula).to_csv(SAVE_DIR + '/copula_recovered_mono_expression.csv', index=False, header=False)
print("Copula CSVs saved.")

# ============================================================
# 3. BIOLOGICAL FIDELITY METRICS — COPULA OUTPUT
# ============================================================
print("\n" + "="*60)
print("BIOLOGICAL FIDELITY METRICS — After Gaussian Copula")
print("="*60)

# ---- METRIC 1: MARKER GENE PRESERVATION ----
print("\n--- Metric 1: Marker Gene Preservation ---")
mean_real_b      = real_b_expr.mean(axis=0)
mean_real_mono   = real_mono_expr.mean(axis=0)
mean_copula_b    = syn_b_copula.mean(axis=0)
mean_copula_mono = syn_mono_copula.mean(axis=0)

r_b,    _ = spearmanr(mean_real_b,    mean_copula_b)
r_mono, _ = spearmanr(mean_real_mono, mean_copula_mono)
print(f"  Spearman r — B class:    {r_b:.4f}")
print(f"  Spearman r — Mono class: {r_mono:.4f}")

n_top = 50
ov_b  = len(set(np.argsort(mean_real_b)[-n_top:])    & set(np.argsort(mean_copula_b)[-n_top:]))    / n_top * 100
ov_m  = len(set(np.argsort(mean_real_mono)[-n_top:]) & set(np.argsort(mean_copula_mono)[-n_top:])) / n_top * 100
print(f"  Top-50 marker overlap — B:    {ov_b:.1f}%")
print(f"  Top-50 marker overlap — Mono: {ov_m:.1f}%")

# ---- METRIC 2: GENE-GENE CORRELATION CONSISTENCY — PBMC ----
print("\n--- Metric 2: Gene-Gene Correlation Consistency ---")

n_top_genes = 100
top_var_idx = np.argsort(real_b_expr.var(axis=0))[-n_top_genes:]

# Remove zero-variance genes from copula output only
valid_b    = syn_b_copula[:,    top_var_idx].var(axis=0) > 0
valid_mono = syn_mono_copula[:, top_var_idx].var(axis=0) > 0
print(f"  Valid genes — B:    {valid_b.sum()} / {n_top_genes}")
print(f"  Valid genes — Mono: {valid_mono.sum()} / {n_top_genes}")

triu_b    = np.triu_indices(valid_b.sum(),    k=1)
triu_mono = np.triu_indices(valid_mono.sum(), k=1)

corr_real_b      = np.corrcoef(real_b_expr[:,    top_var_idx][:, valid_b].T)
corr_copula_b    = np.corrcoef(syn_b_copula[:,   top_var_idx][:, valid_b].T)
corr_real_mono   = np.corrcoef(real_mono_expr[:, top_var_idx][:, valid_mono].T)
corr_copula_mono = np.corrcoef(syn_mono_copula[:, top_var_idx][:, valid_mono].T)

r_cb, _ = pearsonr(corr_real_b[triu_b],       corr_copula_b[triu_b])
r_cm, _ = pearsonr(corr_real_mono[triu_mono],  corr_copula_mono[triu_mono])
print(f"  Pearson r of correlation matrices — B:    {r_cb:.4f}")
print(f"  Pearson r of correlation matrices — Mono: {r_cm:.4f}")

# ---- METRIC 3: DIFFERENTIAL EXPRESSION CONSERVATION ----
print("\n--- Metric 3: Differential Expression Conservation ---")
n_genes = real_b_expr.shape[1]

def compute_DE(expr_b, expr_mono, n_genes):
    pvals = np.ones(n_genes)
    lfc   = np.zeros(n_genes)
    for g in range(n_genes):
        try:
            _, p = mannwhitneyu(expr_b[:, g], expr_mono[:, g], alternative='two-sided')
        except:
            p = 1.0
        pvals[g] = p
        lfc[g]   = np.log2((expr_mono[:, g].mean() + 1e-9) / (expr_b[:, g].mean() + 1e-9))
    return pvals, lfc

print("  Running Wilcoxon on real data...")
real_pv,   real_lfc   = compute_DE(real_b_expr,   real_mono_expr,  n_genes)
print("  Running Wilcoxon on copula data...")
copula_pv, copula_lfc = compute_DE(syn_b_copula,  syn_mono_copula, n_genes)

nlp_real   = -np.log10(real_pv   + 1e-300)
nlp_copula = -np.log10(copula_pv + 1e-300)

r_de, p_de = spearmanr(nlp_real, nlp_copula)
n_top_de   = 100
de_ov      = len(set(np.argsort(nlp_real)[-n_top_de:]) & set(np.argsort(nlp_copula)[-n_top_de:])) / n_top_de * 100
print(f"  Spearman r of -log10(p-values): {r_de:.4f}  (p={p_de:.2e})")
print(f"  Top-100 DE gene overlap:        {de_ov:.1f}%")

# ---- SUMMARY ----
print(f"\n{'='*60}")
print(f"SUMMARY — Gaussian Copula")
print(f"{'='*60}")
print(f"Metric 1 — Marker Gene Preservation:")
print(f"  Spearman r:        B={r_b:.4f}  | Mono={r_mono:.4f}")
print(f"  Top-50 overlap:    B={ov_b:.1f}% | Mono={ov_m:.1f}%")
print(f"Metric 2 — Gene-Gene Correlation Consistency:")
print(f"  Pearson r:         B={r_cb:.4f}  | Mono={r_cm:.4f}")
print(f"Metric 3 — DE Conservation:")
print(f"  Spearman r:        {r_de:.4f}")
print(f"  Top-100 overlap:   {de_ov:.1f}%")

# ============================================================
# 4. COMPARISON TABLE: Before vs After Copula
# ============================================================
print(f"\n{'='*65}")
print("COMPARISON: Option B  vs  Option B + Copula")
print(f"{'='*65}")
print(f"{'Metric':<40} {'Before':>10} {'After':>10}")
print("-"*65)
print(f"{'Marker Spearman r (B)':<40} {'0.8478':>10} {r_b:>10.4f}")
print(f"{'Marker Spearman r (Mono)':<40} {'0.9094':>10} {r_mono:>10.4f}")
print(f"{'Marker Top-50 overlap B (%)':<40} {'84.0':>10} {ov_b:>10.1f}")
print(f"{'Marker Top-50 overlap Mono (%)':<40} {'84.0':>10} {ov_m:>10.1f}")
print(f"{'Corr matrix Pearson r (B)':<40} {'0.0006':>10} {r_cb:>10.4f}")
print(f"{'Corr matrix Pearson r (Mono)':<40} {'0.0029':>10} {r_cm:>10.4f}")
print(f"{'DE Spearman r':<40} {'0.4558':>10} {r_de:>10.4f}")
print(f"{'DE Top-100 overlap (%)':<40} {'60.0':>10} {de_ov:>10.1f}")

'''
# ============================================================
# 5. PDF AND CDF PLOTS — matching your paper style
# ============================================================
plt.rcParams.update({'font.size': 18})

def plot_pdf_cdf(real_data, synthetic_data, title_real, title_synthetic):
    data_min = min(real_data.min(), synthetic_data.min())
    data_max = max(real_data.max(), synthetic_data.max())
    x_vals   = np.linspace(data_min, data_max, 1000)

    # PDF
    plt.figure(figsize=(8, 6))
    kde_real = gaussian_kde(real_data,      bw_method=0.5)
    kde_syn  = gaussian_kde(synthetic_data, bw_method=0.5)

    plt.hist(real_data,      bins=50, range=(data_min, data_max), density=True,
             alpha=0.5, color='black',  edgecolor='black',   label=f'{title_real}')
    plt.hist(synthetic_data, bins=50, range=(data_min, data_max), density=True,
             alpha=0.3, color='gray',   edgecolor='darkgray', label=f'{title_synthetic}')
    plt.plot(x_vals, kde_real(x_vals), linestyle='-',  color='black', linewidth=2, label=f'PDF {title_real}')
    plt.plot(x_vals, kde_syn(x_vals),  linestyle='--', color='gray',  linewidth=2, label=f'PDF {title_synthetic}')

    plt.yscale('log')
    plt.ylim(10**-6, 10**1)
    plt.xticks(np.linspace(data_min, data_max, 5))
    plt.yticks([1e-6, 1e-4, 1e-2, 1e0])
    plt.xlabel('Gene Expression Value', fontsize=18)
    plt.ylabel('Density', fontsize=18)
    plt.legend(fontsize=18)
    plt.grid()
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/pdf_{title_synthetic.replace(' ', '_')}.png", dpi=150, bbox_inches='tight')
    plt.show()

    # CDF
    plt.figure(figsize=(8, 6))
    real_sorted = np.sort(real_data)
    syn_sorted  = np.sort(synthetic_data)
    cdf_real    = np.arange(1, len(real_data) + 1)      / len(real_data)
    cdf_syn     = np.arange(1, len(synthetic_data) + 1) / len(synthetic_data)

    plt.plot(real_sorted, cdf_real, linestyle='dotted', color='black', linewidth=2, label=f'{title_real}')
    plt.plot(syn_sorted,  cdf_syn,  linestyle='--',     color='gray',  linewidth=2, label=f'{title_synthetic}')

    plt.xticks(np.linspace(data_min, data_max, 5))
    plt.yticks(np.linspace(0, 1, 5))
    plt.xlabel('Gene Expression Value', fontsize=18)
    plt.ylabel('Cumulative Probability', fontsize=18)
    plt.legend(fontsize=18)
    plt.grid()
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/cdf_{title_synthetic.replace(' ', '_')}.png", dpi=150, bbox_inches='tight')
    plt.show()

# Flatten for PDF/CDF (all genes, all cells)
print("\nGenerating PDF and CDF plots...")

plot_pdf_cdf(
    real_b_expr.flatten(),
    syn_b_copula.flatten(),
    'Real B', 'Synthetic B (Copula)'
)

plot_pdf_cdf(
    real_mono_expr.flatten(),
    syn_mono_copula.flatten(),
    'Real Mono', 'Synthetic Mono (Copula)'
)

print("\nDone. All plots and CSVs saved.")
'''

Loading data...
real_b:    (1186, 1600)  | real_mono:    (1186, 1600)
syn_b_B:   (1186, 1600)   | syn_mono_B:   (1186, 1600)

Applying Gaussian Copula...
Copula output — syn_b:    (1186, 1600)
Copula output — syn_mono: (1186, 1600)
Copula CSVs saved.

BIOLOGICAL FIDELITY METRICS — After Gaussian Copula

--- Metric 1: Marker Gene Preservation ---
  Spearman r — B class:    0.9897
  Spearman r — Mono class: 0.9931
  Top-50 marker overlap — B:    96.0%
  Top-50 marker overlap — Mono: 98.0%

--- Metric 2: Gene-Gene Correlation Consistency ---
  Valid genes — B:    100 / 100
  Valid genes — Mono: 100 / 100
  Pearson r of correlation matrices — B:    0.6204
  Pearson r of correlation matrices — Mono: 0.6302

--- Metric 3: Differential Expression Conservation ---
  Running Wilcoxon on real data...
  Running Wilcoxon on copula data...
  Spearman r of -log10(p-values): 0.9247  (p=0.00e+00)
  Top-100 DE gene overlap:        96.0%

SUMMARY — Gaussian Copula
Metric 1 — Marker Gene Preservation:
  

'\n# ============================================================\n# 5. PDF AND CDF PLOTS — matching your paper style\n# ============================================================\nplt.rcParams.update({\'font.size\': 18})\n\ndef plot_pdf_cdf(real_data, synthetic_data, title_real, title_synthetic):\n    data_min = min(real_data.min(), synthetic_data.min())\n    data_max = max(real_data.max(), synthetic_data.max())\n    x_vals   = np.linspace(data_min, data_max, 1000)\n\n    # PDF\n    plt.figure(figsize=(8, 6))\n    kde_real = gaussian_kde(real_data,      bw_method=0.5)\n    kde_syn  = gaussian_kde(synthetic_data, bw_method=0.5)\n\n    plt.hist(real_data,      bins=50, range=(data_min, data_max), density=True,\n             alpha=0.5, color=\'black\',  edgecolor=\'black\',   label=f\'{title_real}\')\n    plt.hist(synthetic_data, bins=50, range=(data_min, data_max), density=True,\n             alpha=0.3, color=\'gray\',   edgecolor=\'darkgray\', label=f\'{title_synthetic}\')\n    plt.p

In [ ]:
# Diversity check: variance ratio synthetic vs real
var_ratio_b    = syn_b_copula.var(axis=0).mean() / real_b_expr.var(axis=0).mean()
var_ratio_mono = syn_mono_copula.var(axis=0).mean() / real_mono_expr.var(axis=0).mean()
print(f"Variance ratio B:    {var_ratio_b:.4f}")  # ideally close to 1.0
print(f"Variance ratio Mono: {var_ratio_mono:.4f}")

Variance ratio B:    0.9642
Variance ratio Mono: 1.0366


PDO (Below code will just create new synthetic files)

In [ ]:
# ============================================================
# Generate Synthetic Gene Expression — Corrected Pipeline
# PDO Dataset
# Save to new location
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import tensorflow as tf
import os, glob, re

BASE      = '/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April'
REV_DOCS  = BASE + '/Reverse Calculation Related doc'
SAVE_DIR  = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset'
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Load real genomaps to compute _MIN/_MAX ----
print("Loading real genomaps to compute _MIN/_MAX...")
low_real  = np.load(BASE + '/Real_Low_genomap.npy').astype(np.float32)
high_real = np.load(BASE + '/Real_High_genomap.npy').astype(np.float32)
if low_real.ndim == 3:  low_real  = low_real[...,  np.newaxis]
if high_real.ndim == 3: high_real = high_real[..., np.newaxis]
print(f"Low genomap: {low_real.shape} | High genomap: {high_real.shape}")

stacked = np.vstack([low_real, high_real])
_MIN = float(stacked.min())
_MAX = float(stacked.max())
print(f"_MIN={_MIN:.6f} | _MAX={_MAX:.6f}")

# ---- Load T matrices and stats ----
T_low  = np.load(REV_DOCS + '/transformation_matrix_T_Low_class.npy')
T_high = np.load(REV_DOCS + '/transformation_matrix_T_High_class.npy')
means_low  = np.load(REV_DOCS + '/original_means_Low_class.npy')
means_high = np.load(REV_DOCS + '/original_means_High_class.npy')
stds_low   = np.load(REV_DOCS + '/original_stds_Low_class.npy')
stds_high  = np.load(REV_DOCS + '/original_stds_High_class.npy')

num_genes      = T_low.shape[1]
totalGridPoint = 40 * 40
print(f"num_genes={num_genes} | totalGridPoint={totalGridPoint}")

# ---- Load latest generator checkpoint ----
def load_latest_generator():
    generator_files = glob.glob(BASE + '/Generator Model/generator_Epoch-*.h5')
    if not generator_files:
        print("No checkpoint files found, falling back to generator.h5")
        return tf.keras.models.load_model(BASE + '/generator.h5')
    epochs = [int(re.search(r'Epoch-(\d+)', f).group(1)) for f in generator_files]
    latest_epoch = max(epochs)
    path = BASE + f'/Generator Model/generator_Epoch-{latest_epoch}.h5'
    print(f"Loading generator from epoch {latest_epoch}: {path}")
    return tf.keras.models.load_model(path)

print("\nLoading generator...")
generator = load_latest_generator()

# ---- Generate synthetic genomaps ----
n_low  = low_real.shape[0]   # 1415
n_high = high_real.shape[0]  # 1415
latent_dim = 128

print(f"\nGenerating {n_low} synthetic Low genomaps  (label=-0.5)...")
noise_low  = tf.random.normal(shape=(n_low, latent_dim))
labels_low = np.full((n_low, 1), -0.5, dtype=np.float32)
syn_low_tanh = generator.predict([noise_low, labels_low], batch_size=64)

print(f"Generating {n_high} synthetic High genomaps (label=+0.5)...")
noise_high  = tf.random.normal(shape=(n_high, latent_dim))
labels_high = np.full((n_high, 1), 0.5, dtype=np.float32)
syn_high_tanh = generator.predict([noise_high, labels_high], batch_size=64)

print(f"syn_low_tanh:  {syn_low_tanh.shape}  range=[{syn_low_tanh.min():.3f}, {syn_low_tanh.max():.3f}]")
print(f"syn_high_tanh: {syn_high_tanh.shape} range=[{syn_high_tanh.min():.3f}, {syn_high_tanh.max():.3f}]")

# ---- Reverse min-max normalization ----
def reverse_minmax(syn_tanh, _MIN, _MAX):
    return ((syn_tanh + 1) * (_MAX - _MIN) / 2) + _MIN

syn_low_orig  = reverse_minmax(syn_low_tanh,  _MIN, _MAX)
syn_high_orig = reverse_minmax(syn_high_tanh, _MIN, _MAX)
print(f"\nAfter min-max reversal:")
print(f"  syn_low range:  [{syn_low_orig.min():.4f}, {syn_low_orig.max():.4f}]")
print(f"  syn_high range: [{syn_high_orig.min():.4f}, {syn_high_orig.max():.4f}]")

# ---- Option B: corrected pseudo-inverse reverse ----
def reverse_optionB(syn_orig_scale, T, means, stds):
    n_samples   = syn_orig_scale.shape[0]
    projMat     = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)
    projM = np.zeros((n_samples, num_genes), dtype=np.float32)
    for i in range(n_samples):
        ex = syn_orig_scale[i, :, :, 0].flatten(order='F')
        projM[i, :] = ex[:num_genes]
    X_norm = np.matmul(projM, projMat_inv)
    return ((X_norm * stds) + means).astype(np.float32)

print("\nApplying corrected pseudo-inverse reverse...")
syn_low_expr  = reverse_optionB(syn_low_orig,  T_low,  means_low,  stds_low)
syn_high_expr = reverse_optionB(syn_high_orig, T_high, means_high, stds_high)

print(f"syn_low_expr:  {syn_low_expr.shape}  range=[{syn_low_expr.min():.3f}, {syn_low_expr.max():.3f}]")
print(f"syn_high_expr: {syn_high_expr.shape} range=[{syn_high_expr.min():.3f}, {syn_high_expr.max():.3f}]")

# ---- Save ----
low_save_path  = SAVE_DIR + '/new_recovered_low_expression.csv'
high_save_path = SAVE_DIR + '/new_recovered_high_expression.csv'

pd.DataFrame(syn_low_expr).to_csv( low_save_path,  index=False, header=False, float_format='%.6f')
pd.DataFrame(syn_high_expr).to_csv(high_save_path, index=False, header=False, float_format='%.6f')

print(f"\nSaved:")
print(f"  {low_save_path}")
print(f"  {high_save_path}")
print("Done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading real genomaps to compute _MIN/_MAX...
Low genomap: (1415, 40, 40, 1) | High genomap: (1415, 40, 40, 1)
_MIN=-0.738665 | _MAX=37.589901
num_genes=1600 | totalGridPoint=1600

Loading generator...
Loading generator from epoch 2995: /content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April/Generator Model/generator_Epoch-2995.h5



Generating 1415 synthetic Low genomaps  (label=-0.5)...
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 139ms/step
Generating 1415 synthetic High genomaps (label=+0.5)...
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step
syn_low_tanh:  (1415, 40, 40, 1)  range=[-0.998, 0.999]
syn_high_tanh: (1415, 40, 40, 1) range=[-0.998, 0.993]

After min-max reversal:
  syn_low range:  [-0.6944, 37.5769]
  syn_high range: [-0.7040, 37.4641]

Applying corrected pseudo-inverse reverse...
syn_low_expr:  (1415, 1600)  range=[-0.213, 23.831]
syn_high_expr: (1415, 1600) range=[-0.827, 73.627]

Saved:
  /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/new_recovered_low_expression.csv
  /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/new_recovered_high_expression.csv
Done.


PDO: Gaussian Capula Study

In [ ]:
# ============================================================
# Gaussian Copula Post-processing + 3 Metrics + PDF/CDF
# PDO Dataset
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, spearmanr, pearsonr, mannwhitneyu
from scipy.stats import gaussian_kde
import os, warnings
warnings.filterwarnings('ignore')

BASE      = '/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April'
SAVE_DIR  = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset'
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# 1. LOAD DATA
# ============================================================
print("Loading data...")
real_low_expr  = pd.read_csv(BASE + '/3. Differential_Low_Raw_Finalized.csv',  header=None).values.astype(np.float32)
real_high_expr = pd.read_csv(BASE + '/3. Stem_High_Raw_Finalized.csv', header=None).values.astype(np.float32)

syn_low_expr  = pd.read_csv(SAVE_DIR + '/new_recovered_low_expression.csv',  header=None).values.astype(np.float32)
syn_high_expr = pd.read_csv(SAVE_DIR + '/new_recovered_high_expression.csv', header=None).values.astype(np.float32)

print(f"real_low:  {real_low_expr.shape}  | real_high:  {real_high_expr.shape}")
print(f"syn_low:   {syn_low_expr.shape}   | syn_high:   {syn_high_expr.shape}")

Loading data...
real_low:  (1415, 1600)  | real_high:  (1415, 1600)
syn_low:   (1415, 1600)   | syn_high:   (1415, 1600)


In [ ]:
# ============================================================
# 2. GAUSSIAN COPULA POST-PROCESSING
# ============================================================
print("\nApplying Gaussian Copula...")

def apply_gaussian_copula(syn_expr, real_expr):
    n_syn, n_genes = syn_expr.shape

    # Step 1: Rank-transform synthetic → uniform pseudo-observations
    ranks = np.argsort(np.argsort(syn_expr, axis=0), axis=0)
    u = (ranks + 1) / (n_syn + 1)

    # Step 2: Map uniform → standard Gaussian
    z = norm.ppf(u)

    # Step 3: Compute real correlation matrix and Cholesky decomposition
    R_real = np.corrcoef(real_expr.T)
    R_real += np.eye(n_genes) * 1e-4
    L = np.linalg.cholesky(R_real)

    # Standardize z
    z = (z - z.mean(axis=0)) / (z.std(axis=0) + 1e-8)

    # Apply Cholesky to impose real correlation structure
    z_corr = z @ L.T

    # Step 4: Map back to real marginals via quantile matching
    u_corr = norm.cdf(z_corr)
    u_corr = np.clip(u_corr, 0.001, 0.999)

    syn_copula = np.zeros_like(syn_expr)
    for g in range(n_genes):
        syn_copula[:, g] = np.quantile(real_expr[:, g], u_corr[:, g])

    return syn_copula.astype(np.float32)

syn_low_copula  = apply_gaussian_copula(syn_low_expr,  real_low_expr)
syn_high_copula = apply_gaussian_copula(syn_high_expr, real_high_expr)

print(f"Copula output — syn_low:  {syn_low_copula.shape}")
print(f"Copula output — syn_high: {syn_high_copula.shape}")

# Save
pd.DataFrame(syn_low_copula).to_csv( SAVE_DIR + '/copula_recovered_low_expression.csv',  index=False, header=False, float_format='%.6f')
pd.DataFrame(syn_high_copula).to_csv(SAVE_DIR + '/copula_recovered_high_expression.csv', index=False, header=False, float_format='%.6f')
print("Copula CSVs saved.")

# ============================================================
# 3. BIOLOGICAL FIDELITY METRICS
# ============================================================
print("\n" + "="*60)
print("BIOLOGICAL FIDELITY METRICS — After Gaussian Copula (PDO)")
print("="*60)

# ---- METRIC 1: MARKER GENE PRESERVATION ----
print("\n--- Metric 1: Marker Gene Preservation ---")
mean_real_low    = real_low_expr.mean(axis=0)
mean_real_high   = real_high_expr.mean(axis=0)
mean_copula_low  = syn_low_copula.mean(axis=0)
mean_copula_high = syn_high_copula.mean(axis=0)

r_low,  _ = spearmanr(mean_real_low,  mean_copula_low)
r_high, _ = spearmanr(mean_real_high, mean_copula_high)
print(f"  Spearman r — Low (Differentiated): {r_low:.4f}")
print(f"  Spearman r — High (Stem):          {r_high:.4f}")

n_top = 50
ov_low  = len(set(np.argsort(mean_real_low)[-n_top:])  & set(np.argsort(mean_copula_low)[-n_top:]))  / n_top * 100
ov_high = len(set(np.argsort(mean_real_high)[-n_top:]) & set(np.argsort(mean_copula_high)[-n_top:])) / n_top * 100
print(f"  Top-50 marker overlap — Low:  {ov_low:.1f}%")
print(f"  Top-50 marker overlap — High: {ov_high:.1f}%")

# ---- METRIC 2: GENE-GENE CORRELATION CONSISTENCY ----

print("\n--- Metric 2: Gene-Gene Correlation Consistency ---")

n_top_genes = 100
top_var_idx = np.argsort(real_low_expr.var(axis=0))[-n_top_genes:]

# Remove zero-variance genes from copula output only
valid_low  = syn_low_copula[:,  top_var_idx].var(axis=0) > 0
valid_high = syn_high_copula[:, top_var_idx].var(axis=0) > 0
print(f"  Valid genes — Low:  {valid_low.sum()} / {n_top_genes}")
print(f"  Valid genes — High: {valid_high.sum()} / {n_top_genes}")

triu_low  = np.triu_indices(valid_low.sum(),  k=1)
triu_high = np.triu_indices(valid_high.sum(), k=1)

corr_real_low    = np.corrcoef(real_low_expr[:,   top_var_idx][:, valid_low].T)
corr_copula_low  = np.corrcoef(syn_low_copula[:,  top_var_idx][:, valid_low].T)
corr_real_high   = np.corrcoef(real_high_expr[:,  top_var_idx][:, valid_high].T)
corr_copula_high = np.corrcoef(syn_high_copula[:, top_var_idx][:, valid_high].T)

r_cl, _ = pearsonr(corr_real_low[triu_low],   corr_copula_low[triu_low])
r_ch, _ = pearsonr(corr_real_high[triu_high], corr_copula_high[triu_high])
print(f"  Pearson r of correlation matrices — Low:  {r_cl:.4f}")
print(f"  Pearson r of correlation matrices — High: {r_ch:.4f}")

# ---- METRIC 3: DIFFERENTIAL EXPRESSION CONSERVATION ----
print("\n--- Metric 3: Differential Expression Conservation ---")
n_genes = real_low_expr.shape[1]

def compute_DE(expr_low, expr_high, n_genes):
    pvals = np.ones(n_genes)
    lfc   = np.zeros(n_genes)
    for g in range(n_genes):
        try:
            _, p = mannwhitneyu(expr_low[:, g], expr_high[:, g], alternative='two-sided')
        except:
            p = 1.0
        pvals[g] = p
        lfc[g]   = np.log2((expr_high[:, g].mean() + 1e-9) / (expr_low[:, g].mean() + 1e-9))
    return pvals, lfc

print("  Running Wilcoxon on real data...")
real_pv,   real_lfc   = compute_DE(real_low_expr,  real_high_expr,  n_genes)
print("  Running Wilcoxon on copula data...")
copula_pv, copula_lfc = compute_DE(syn_low_copula, syn_high_copula, n_genes)

nlp_real   = -np.log10(real_pv   + 1e-300)
nlp_copula = -np.log10(copula_pv + 1e-300)

r_de, p_de = spearmanr(nlp_real, nlp_copula)
n_top_de   = 100
de_ov      = len(set(np.argsort(nlp_real)[-n_top_de:]) & set(np.argsort(nlp_copula)[-n_top_de:])) / n_top_de * 100
print(f"  Spearman r of -log10(p-values): {r_de:.4f}  (p={p_de:.2e})")
print(f"  Top-100 DE gene overlap:        {de_ov:.1f}%")

# ---- SUMMARY ----
print(f"\n{'='*60}")
print(f"SUMMARY — Gaussian Copula (PDO)")
print(f"{'='*60}")
print(f"Metric 1 — Marker Gene Preservation:")
print(f"  Spearman r:     Low={r_low:.4f}  | High={r_high:.4f}")
print(f"  Top-50 overlap: Low={ov_low:.1f}% | High={ov_high:.1f}%")
print(f"Metric 2 — Gene-Gene Correlation Consistency:")
print(f"  Pearson r:      Low={r_cl:.4f}  | High={r_ch:.4f}")
print(f"Metric 3 — DE Conservation:")
print(f"  Spearman r:     {r_de:.4f}")
print(f"  Top-100 overlap: {de_ov:.1f}%")

# ============================================================
# 4. PDF AND CDF PLOTS — matching paper style
# ============================================================
#plt.rcParams.update({'font.size': 18})

#def plot_pdf_cdf(real_data, synthetic_data, title_real, title_synthetic, save_dir):
#    data_min = min(real_data.min(), synthetic_data.min())
#    data_max = max(real_data.max(), synthetic_data.max())
#    x_vals   = np.linspace(data_min, data_max, 1000)
#
#    # PDF
#    plt.figure(figsize=(8, 6))
#    kde_real = gaussian_kde(real_data,      bw_method=0.5)
#    kde_syn  = gaussian_kde(synthetic_data, bw_method=0.5)
#

#   plt.hist(real_data,      bins=50, range=(data_min, data_max), density=True,
#             alpha=0.5, color='black',  edgecolor='black',    label=f'{title_real}')
#    plt.hist(synthetic_data, bins=50, range=(data_min, data_max), density=True,
#             alpha=0.3, color='gray',   edgecolor='darkgray', label=f'{title_synthetic}')
#    plt.plot(x_vals, kde_real(x_vals), linestyle='-',  color='black', linewidth=2, label=f'PDF {title_real}')
#    plt.plot(x_vals, kde_syn(x_vals),  linestyle='--', color='gray',  linewidth=2, label=f'PDF {title_synthetic}')

#    plt.yscale('log')
#    plt.ylim(10**-6, 10**1)
#    plt.xticks(np.linspace(data_min, data_max, 5))
#    plt.yticks([1e-6, 1e-4, 1e-2, 1e0])
#    plt.xlabel('Gene Expression Value', fontsize=18)
#    plt.ylabel('Density', fontsize=18)
#    plt.legend(fontsize=18)
#    plt.grid()
##    plt.tight_layout()
 #   plt.savefig(f"{save_dir}/pdf_{title_synthetic.replace(' ', '_')}.png", dpi=150, bbox_inches='tight')
  #  plt.show()

    # CDF
#    plt.figure(figsize=(8, 6))
#    real_sorted = np.sort(real_data)
#    syn_sorted  = np.sort(synthetic_data)
#    cdf_real    = np.arange(1, len(real_data) + 1)      / len(real_data)
#    cdf_syn     = np.arange(1, len(synthetic_data) + 1) / len(synthetic_data)

#    plt.plot(real_sorted, cdf_real, linestyle='dotted', color='black', linewidth=2, label=f'{title_real}')
#    plt.plot(syn_sorted,  cdf_syn,  linestyle='--',     color='gray',  linewidth=2, label=f'{title_synthetic}')

#    plt.xticks(np.linspace(data_min, data_max, 5))
##    plt.yticks(np.linspace(0, 1, 5))
#    plt.xlabel('Gene Expression Value', fontsize=18)
#    plt.ylabel('Cumulative Probability', fontsize=18)
#    plt.legend(fontsize=18)
#    plt.grid()
#    plt.tight_layout()
#    plt.savefig(f"{save_dir}/cdf_{title_synthetic.replace(' ', '_')}.png", dpi=150, bbox_inches='tight')
#    plt.show()

#print("\nGenerating PDF and CDF plots...")

#plot_pdf_cdf(
#    real_low_expr.flatten(),
##    syn_low_copula.flatten(),
#    'Real Low (Differentiated)', 'Synthetic Low (Copula)',
#    SAVE_DIR
#)

#plot_pdf_cdf(
#    real_high_expr.flatten(),
#    syn_high_copula.flatten(),
#    'Real High (Stem)', 'Synthetic High (Copula)',
#    SAVE_DIR
#)
#
#print("\nDone. All plots and CSVs saved.")


Applying Gaussian Copula...
Copula output — syn_low:  (1415, 1600)
Copula output — syn_high: (1415, 1600)
Copula CSVs saved.

BIOLOGICAL FIDELITY METRICS — After Gaussian Copula (PDO)

--- Metric 1: Marker Gene Preservation ---
  Spearman r — Low (Differentiated): 0.8766
  Spearman r — High (Stem):          0.9670
  Top-50 marker overlap — Low:  74.0%
  Top-50 marker overlap — High: 92.0%

--- Metric 2: Gene-Gene Correlation Consistency ---
  Valid genes — Low:  97 / 100
  Valid genes — High: 100 / 100
  Pearson r of correlation matrices — Low:  0.3121
  Pearson r of correlation matrices — High: 0.6796

--- Metric 3: Differential Expression Conservation ---
  Running Wilcoxon on real data...
  Running Wilcoxon on copula data...
  Spearman r of -log10(p-values): 0.9361  (p=0.00e+00)
  Top-100 DE gene overlap:        87.0%

SUMMARY — Gaussian Copula (PDO)
Metric 1 — Marker Gene Preservation:
  Spearman r:     Low=0.8766  | High=0.9670
  Top-50 overlap: Low=74.0% | High=92.0%
Metric 2 — 

In [ ]:
var_ratio_low  = syn_low_copula.var(axis=0).mean()  / real_low_expr.var(axis=0).mean()
var_ratio_high = syn_high_copula.var(axis=0).mean() / real_high_expr.var(axis=0).mean()
print(f"Variance ratio Low:  {var_ratio_low:.4f}")
print(f"Variance ratio High: {var_ratio_high:.4f}")

Variance ratio Low:  0.5778
Variance ratio High: 2.3811


In [ ]:
# Check variance ratio BEFORE copula (Option B output)
var_ratio_low_B  = syn_low_expr.var(axis=0).mean()  / real_low_expr.var(axis=0).mean()
var_ratio_high_B = syn_high_expr.var(axis=0).mean() / real_high_expr.var(axis=0).mean()
print(f"Before copula — Variance ratio Low:  {var_ratio_low_B:.4f}")
print(f"Before copula — Variance ratio High: {var_ratio_high_B:.4f}")

Before copula — Variance ratio Low:  1.1327
Before copula — Variance ratio High: 0.9914


copula effect on PDO
The copula does quantile matching — it maps synthetic values to real data quantiles. For the PDO Low class (Differentiated), the real data is very sparse with many zeros. When the copula forces the synthetic values to match those real quantiles, many genes get collapsed to zero. This kills the variance for the Low class.
For the High class the opposite — the real Stem data has higher expression values, so the copula stretches the synthetic values upward, inflating variance.
The cGAN itself is generating diverse, well-balanced samples. The copula is breaking it for PDO.

This changes the picture significantly
Before CopulaAfter Copula
Variance ratio Low 1.13✓ 0.58✗
Variance ratio High 0.99✓ 2.38✗
Gene-gene corr Low ~00.31
Gene-gene corr High ~00.68
The copula fixes correlation but breaks diversity for PDO. We have to trade one problem for another on the PDO dataset.

Honest conclusion
For PBMC — the copula is a clear win. Diversity preserved (0.96/1.04), all metrics strong.
For PDO — the copula improves correlation but at the cost of diversity. The cGAN output before copula is actually more faithful in terms of diversity. This is a genuine limitation driven by PDO sparsity.

What to do
Option 1 — Apply copula to PBMC only, skip it for PDO
Report PBMC with copula and PDO without. Be transparent about why — PDO sparsity makes quantile matching inappropriate.
Option 2 — Apply a softer version for PDO
Instead of full quantile matching, use only the correlation step without forcing marginals to match real data. This would preserve diversity while still improving correlation.
Option 3 — Report both before and after for PDO
Show the trade-off transparently — before copula has better diversity, after copula has better correlation. Let the reader see the full picture.
My recommendation is Option 1 — it is the cleanest and most honest. PBMC benefits from the copula. PDO does not. Applying different post-processing to different datasets based on their properties is scientifically justified and shows you understand your data.

Discussion: Key Observations
Strengths across both datasets:

Metrics 1 and 3 are consistently strong — marker gene preservation and DE conservation are excellent
PBMC outperforms PDO on all metrics — expected, PBMC has distinct cell types and richer data

PDO Low class is the weakest point — Metric 2 at 0.31 and Top-50 marker overlap at 74%. Both are explained by the same reason — PDO Low (Differentiated) is the sparsest class in your dataset with the most subtle transcriptional signature
The biological explanation is consistent — the paper already states PDO data is sparse with subtle transcriptional differences. The results reflect exactly that.
These are solid, defensible numbers. Shall we now write the response letter to the reviewer and the updated results section?

Debug: NaN issue in PDO

In [ ]:
# Run this to understand exactly what's happening
import numpy as np

R_low = np.corrcoef(real_low_expr.T)
print(f"Any NaN in R_low: {np.isnan(R_low).any()}")
print(f"Min eigenvalue:   {np.linalg.eigvalsh(R_low).min():.6f}")



R_low = np.corrcoef(real_low_expr.T)
print(f"R_low shape: {R_low.shape}")
print(f"Any NaN in R_low: {np.isnan(R_low).any()}")
print(f"Any Inf in R_low: {np.isinf(R_low).any()}")

# After make_positive_definite
def make_positive_definite(R):
    eigvals, eigvecs = np.linalg.eigh(R)
    eigvals = np.clip(eigvals, 1e-6, None)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T

R_fixed = make_positive_definite(R_low)
print(f"Min eigenvalue after fix: {np.linalg.eigvalsh(R_fixed).min():.8f}")

# Try Cholesky on fixed matrix
try:
    L = np.linalg.cholesky(R_fixed)
    print("Cholesky succeeded")
except Exception as e:
    print(f"Cholesky failed: {e}")

# Check the copula output for Low class specifically
syn_low_ranks = np.argsort(np.argsort(syn_low_expr, axis=0), axis=0)
u_low = (syn_low_ranks + 1) / (syn_low_expr.shape[0] + 1)
from scipy.stats import norm
z_low = norm.ppf(u_low)
print(f"Any NaN in z_low: {np.isnan(z_low).any()}")
print(f"Any Inf in z_low: {np.isinf(z_low).any()}")
z_low_std = (z_low - z_low.mean(axis=0)) / (z_low.std(axis=0) + 1e-8)
z_corr_low = z_low_std @ L.T
print(f"Any NaN in z_corr_low: {np.isnan(z_corr_low).any()}")
u_corr_low = norm.cdf(z_corr_low)
print(f"Any NaN in u_corr_low: {np.isnan(u_corr_low).any()}")

# Check correlation of copula output
corr_copula_low = np.corrcoef(syn_low_copula[:, np.argsort(real_low_expr.var(axis=0))[-100:]].T)
print(f"Any NaN in corr_copula_low: {np.isnan(corr_copula_low).any()}")
triu = np.triu_indices(100, k=1)
print(f"Any NaN in upper triangle: {np.isnan(corr_copula_low[triu]).any()}")

Any NaN in R_low: False
Min eigenvalue:   -0.000000
R_low shape: (1600, 1600)
Any NaN in R_low: False
Any Inf in R_low: False
Min eigenvalue after fix: 0.00000100
Cholesky succeeded
Any NaN in z_low: False
Any Inf in z_low: False
Any NaN in z_corr_low: False
Any NaN in u_corr_low: False
Any NaN in corr_copula_low: True
Any NaN in upper triangle: True


PCA - t-SNE - UMAP

In [ ]:
# ============================================================
# UMAP + t-SNE: Real vs Synthetic (Before & After Copula)
# Both PBMC and PDO
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import os, warnings
warnings.filterwarnings('ignore')

# Install umap if needed
# !pip install umap-learn --quiet

PBMC_BASE  = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
PBMC_SAVE  = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset'
PDO_BASE   = '/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April'
PDO_SAVE   = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset'
PLOT_DIR   = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# ============================================================
# 1. LOAD ALL DATA
# ============================================================
print("Loading PBMC data...")
real_b_expr    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PBMC dataset/b_Class_dataset.csv',    header=None).values.astype(np.float32)
real_mono_expr = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PBMC dataset/mono_Class_dataset.csv', header=None).values.astype(np.float32)
syn_b_before   = pd.read_csv(PBMC_SAVE + '/new_recovered_b_expression.csv',    header=None).values.astype(np.float32)
syn_mono_before= pd.read_csv(PBMC_SAVE + '/new_recovered_mono_expression.csv', header=None).values.astype(np.float32)
syn_b_after    = pd.read_csv(PBMC_SAVE + '/copula_recovered_b_expression.csv',    header=None).values.astype(np.float32)
syn_mono_after = pd.read_csv(PBMC_SAVE + '/copula_recovered_mono_expression.csv', header=None).values.astype(np.float32)
print(f"  real_b: {real_b_expr.shape} | real_mono: {real_mono_expr.shape}")

print("Loading PDO data...")
real_low_expr  = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PDO dataset/3. Differential_Low_Raw_Finalized.csv',  header=None).values.astype(np.float32)
real_high_expr = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PDO dataset/3. Stem_High_Raw_Finalized.csv', header=None).values.astype(np.float32)
syn_low_before = pd.read_csv(PDO_SAVE + '/new_recovered_low_expression.csv',  header=None).values.astype(np.float32)
syn_high_before= pd.read_csv(PDO_SAVE + '/new_recovered_high_expression.csv', header=None).values.astype(np.float32)
syn_low_after  = pd.read_csv(PDO_SAVE + '/copula_recovered_low_expression.csv',  header=None).values.astype(np.float32)
syn_high_after = pd.read_csv(PDO_SAVE + '/copula_recovered_high_expression.csv', header=None).values.astype(np.float32)
print(f"  real_low: {real_low_expr.shape} | real_high: {real_high_expr.shape}")

# ============================================================
# 2. HELPER: PCA → UMAP + t-SNE PLOTS
# ============================================================

def plot_umap_tsne(real_class1, real_class2,
                   syn_class1_before, syn_class2_before,
                   syn_class1_after,  syn_class2_after,
                   label_class1, label_class2,
                   dataset_name, save_dir,
                   n_pca=50, random_state=42):
    """
    For each of 2 conditions (before/after copula):
    - Combine real class1, real class2, syn class1, syn class2
    - Run PCA → UMAP and t-SNE
    - Plot with 4 colors: real class1, real class2, syn class1, syn class2
    """

    colors = {
        f'Real {label_class1}':      '#2166ac',   # dark blue
        f'Real {label_class2}':      '#d6604d',   # dark red
        f'Synthetic {label_class1}': '#92c5de',   # light blue
        f'Synthetic {label_class2}': '#f4a582',   # light red/orange
    }

    for condition, syn1, syn2 in [
        ('Before Copula', syn_class1_before, syn_class2_before),
        ('After Copula',  syn_class1_after,  syn_class2_after)
    ]:
        print(f"\nProcessing {dataset_name} — {condition}...")

        # Stack all data
        X = np.vstack([real_class1, real_class2, syn1, syn2])
        labels = (
            [f'Real {label_class1}']      * len(real_class1) +
            [f'Real {label_class2}']      * len(real_class2) +
            [f'Synthetic {label_class1}'] * len(syn1) +
            [f'Synthetic {label_class2}'] * len(syn2)
        )
        labels = np.array(labels)

        # Standardize
        X_scaled = StandardScaler().fit_transform(X)

        # PCA first (speeds up UMAP and t-SNE)
        n_components = min(n_pca, X_scaled.shape[0]-1, X_scaled.shape[1])
        print(f"  Running PCA ({n_components} components)...")
        X_pca = PCA(n_components=n_components, random_state=random_state).fit_transform(X_scaled)

        # UMAP
        print(f"  Running UMAP...")
        reducer = umap.UMAP(n_neighbors=30, min_dist=0.3,
                            n_components=2, random_state=random_state)
        X_umap = reducer.fit_transform(X_pca)

        # t-SNE
        print(f"  Running t-SNE...")
        X_tsne = TSNE(n_components=2, perplexity=40,
                      random_state=random_state, n_iter=1000).fit_transform(X_pca)

        # Plot both
        for embedding, method in [(X_umap, 'UMAP'), (X_tsne, 't-SNE')]:
            fig, ax = plt.subplots(figsize=(8, 7))

            for label in [f'Real {label_class1}', f'Real {label_class2}',
                          f'Synthetic {label_class1}', f'Synthetic {label_class2}']:
                mask = labels == label
                # Real data: filled circles, Synthetic: open circles
                marker = 'o' if 'Real' in label else '^'
                alpha  = 0.7 if 'Real' in label else 0.5
                size   = 8   if 'Real' in label else 6
                ax.scatter(embedding[mask, 0], embedding[mask, 1],
                           c=colors[label], label=label,
                           marker=marker, alpha=alpha, s=size, linewidths=0)

            ax.set_title(f'{dataset_name} — {method} — {condition}', fontsize=13, fontweight='bold')
            ax.set_xlabel(f'{method} 1', fontsize=12)
            ax.set_ylabel(f'{method} 2', fontsize=12)
            ax.legend(fontsize=10, markerscale=2, framealpha=0.8)
            ax.set_xticks([]); ax.set_yticks([])
            plt.tight_layout()

            fname = f'{dataset_name}_{method}_{condition.replace(" ", "_")}.png'
            plt.savefig(f'{save_dir}/{fname}', dpi=150, bbox_inches='tight')
            plt.show()
            print(f"  Saved: {fname}")

# ============================================================
# 3. RUN FOR PBMC
# ============================================================
print("\n" + "="*60)
print("PBMC PLOTS")
print("="*60)
plot_umap_tsne(
    real_b_expr,    real_mono_expr,
    syn_b_before,   syn_mono_before,
    syn_b_after,    syn_mono_after,
    label_class1='B', label_class2='Mono',
    dataset_name='PBMC',
    save_dir=PLOT_DIR
)

# ============================================================
# 4. RUN FOR PDO
# ============================================================
print("\n" + "="*60)
print("PDO PLOTS")
print("="*60)
plot_umap_tsne(
    real_low_expr,   real_high_expr,
    syn_low_before,  syn_high_before,
    syn_low_after,   syn_high_after,
    label_class1='Low (Differentiated)', label_class2='High (Stem)',
    dataset_name='PDO',
    save_dir=PLOT_DIR
)

print("\nAll plots saved to:", PLOT_DIR)

Loading PBMC data...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/copula_recovered_b_expression.csv'